# 03_Note_Extraction

## Objective

The goal of this notebook is to extract clinical notes for the final study cohort created in the previous notebook.

This notebook will:

- load the cleaned `sepsis_cohort.csv`
- load `NOTEEVENTS.csv`
- keep only notes linked to patients in the cohort
- remove error notes
- create note timestamps using `CHARTTIME` and `CHARTDATE`
- select notes written within the first 24 hours of ICU admission
- combine notes per hospital admission
- save the final note dataset for text preprocessing and modeling

In [1]:
# Code Starter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"
OUTPUT_PATH = "../outputs/"

## Load data

In [4]:
# Load cohort
cohort = pd.read_csv(PROCESSED_PATH + "sepsis_cohort.csv")

print("Cohort shape:", cohort.shape)
cohort.head()

Cohort shape: (38509, 14)


,SUBJECT_ID,HADM_ID,ICUSTAY_ID,ADMITTIME,DISCHTIME,INTIME,OUTTIME,LOS,GENDER,DOB,AGE,DIAGNOSIS,HOSPITAL_EXPIRE_FLAG,SEPSIS_LABEL
0,3,145834,211552,2101-10-20 19:08:00,2101-10-31 13:58:00,2101-10-20 19:10:11,2101-10-26 20:43:09,6.0646,M,2025-04-11,76,HYPOTENSION,0,1
1,4,185777,294638,2191-03-16 00:28:00,2191-03-23 18:41:00,2191-03-16 00:29:31,2191-03-17 16:46:31,1.6785,F,2143-05-12,47,"FEVER,DEHYDRATION,FAILURE TO THRIVE",0,0
2,6,107064,228232,2175-05-30 07:15:00,2175-06-15 16:00:00,2175-05-30 21:30:54,2175-06-03 13:39:54,3.6729,F,2109-06-21,65,CHRONIC RENAL FAILURE/SDA,0,0
3,9,150750,220597,2149-11-09 13:06:00,2149-11-14 10:15:00,2149-11-09 13:07:02,2149-11-14 20:52:14,5.3231,M,2108-01-26,41,HEMORRHAGIC CVA,1,0
4,11,194540,229441,2178-04-16 06:18:00,2178-05-11 19:00:00,2178-04-16 06:19:32,2178-04-17 20:21:05,1.5844,F,2128-02-22,50,BRAIN MASS,0,0


In [6]:
# Load NOTEEVENTS
notes = pd.read_csv(DATA_PATH + "NOTEEVENTS.csv")

print("NOTEEVENTS shape:", notes.shape)
notes.head()

NOTEEVENTS shape: (2083180, 11)


,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT
0,174,22532,167853.0,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...
1,175,13702,107527.0,2118-06-14,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2118-6-2**] Discharg...
2,176,13702,167118.0,2119-05-25,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2119-5-4**] D...
3,177,13702,196489.0,2124-08-18,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2124-7-21**] ...
4,178,26880,135453.0,2162-03-25,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2162-3-3**] D...


In [8]:
# Convert dates
cohort["INTIME"] = pd.to_datetime(cohort["INTIME"])

notes["CHARTDATE"] = pd.to_datetime(notes["CHARTDATE"], errors="coerce")
notes["CHARTTIME"] = pd.to_datetime(notes["CHARTTIME"], errors="coerce")

## Step 1: Keep notes for cohort admissions only

The notes table is filtered using `SUBJECT_ID` and `HADM_ID` so that only notes belonging to patients in the final cohort are retained.

In [11]:
notes_cohort = notes.merge(
    cohort[["SUBJECT_ID", "HADM_ID", "ICUSTAY_ID", "INTIME", "SEPSIS_LABEL"]],
    on=["SUBJECT_ID", "HADM_ID"],
    how="inner"
)

print("Notes linked to cohort:", notes_cohort.shape)
notes_cohort.head()

Notes linked to cohort: (1085693, 14)


,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT,ICUSTAY_ID,INTIME,SEPSIS_LABEL
0,174,22532,167853.0,2151-08-04,NaT,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,276393,2151-07-16 14:31:00,0
1,175,13702,107527.0,2118-06-14,NaT,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2118-6-2**] Discharg...,239416,2118-06-02 19:20:00,0
2,178,26880,135453.0,2162-03-25,NaT,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2162-3-3**] D...,201964,2162-03-03 18:46:34,0
3,179,53181,170490.0,2172-03-08,NaT,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2172-3-5**] D...,206021,2172-03-05 14:00:16,0
4,180,20646,134727.0,2112-12-10,NaT,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2112-12-8**] ...,210303,2112-12-08 19:58:48,0


## Step 2: Remove error notes

Rows marked as error notes are removed to avoid using incorrect clinical documentation.

In [14]:
# Remove error notes if ISERROR is available
notes_cohort = notes_cohort[notes_cohort["ISERROR"].isna()].copy()

print("After removing error notes:", notes_cohort.shape)

After removing error notes: (1085086, 14)


## Step 3: Create note timestamp

`CHARTTIME` is used when available. If `CHARTTIME` is missing, `CHARTDATE` is used as a fallback timestamp.

In [17]:
notes_cohort["NOTE_TIME"] = notes_cohort["CHARTTIME"].fillna(notes_cohort["CHARTDATE"])

notes_cohort[["SUBJECT_ID", "HADM_ID", "INTIME", "NOTE_TIME", "CATEGORY", "TEXT"]].head()

,SUBJECT_ID,HADM_ID,INTIME,NOTE_TIME,CATEGORY,TEXT
0,22532,167853.0,2151-07-16 14:31:00,2151-08-04,Discharge summary,Admission Date: [**2151-7-16**] Dischar...
1,13702,107527.0,2118-06-02 19:20:00,2118-06-14,Discharge summary,Admission Date: [**2118-6-2**] Discharg...
2,26880,135453.0,2162-03-03 18:46:34,2162-03-25,Discharge summary,Admission Date: [**2162-3-3**] D...
3,53181,170490.0,2172-03-05 14:00:16,2172-03-08,Discharge summary,Admission Date: [**2172-3-5**] D...
4,20646,134727.0,2112-12-08 19:58:48,2112-12-10,Discharge summary,Admission Date: [**2112-12-8**] ...


## Step 4: Select notes from the first 24 hours of ICU admission

Only notes written between ICU admission time (`INTIME`) and 24 hours after ICU admission are retained.

In [20]:
notes_cohort["TIME_FROM_ICU_ADMIT"] = notes_cohort["NOTE_TIME"] - notes_cohort["INTIME"]

first24h_notes = notes_cohort[
    (notes_cohort["TIME_FROM_ICU_ADMIT"] >= pd.Timedelta(hours=0)) &
    (notes_cohort["TIME_FROM_ICU_ADMIT"] <= pd.Timedelta(hours=24))
].copy()

print("First 24h notes shape:", first24h_notes.shape)
first24h_notes.head()

First 24h notes shape: (215508, 16)


,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT,ICUSTAY_ID,INTIME,SEPSIS_LABEL,NOTE_TIME,TIME_FROM_ICU_ADMIT
29,249,62745,133623.0,2145-12-01,NaT,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2145-11-30**] ...,269020,2145-11-30 22:33:04,0,2145-12-01,0 days 01:26:56
38,201,4127,167565.0,2193-05-31,NaT,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2193-5-30**] Dischar...,261181,2193-05-30 04:23:47,0,2193-05-31,0 days 19:36:13
79,51,57276,171108.0,2118-07-11,NaT,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2118-7-10**] ...,264166,2118-07-10 04:16:53,0,2118-07-11,0 days 19:43:07
112,105,20649,177831.0,2154-12-15,NaT,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2154-12-14**] Death Da...,297925,2154-12-14 00:42:00,0,2154-12-15,0 days 23:18:00
115,108,24622,192777.0,2158-09-18,NaT,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2158-9-14**] Dischar...,235285,2158-09-17 07:54:34,0,2158-09-18,0 days 16:05:26


## Step 5: Inspect note categories

The distribution of note categories is examined to understand the types of notes available within the first 24 hours.

In [23]:
first24h_notes["CATEGORY"].value_counts()

CATEGORY
Nursing/other        63531
Radiology            55739
Nursing              40394
Physician            27209
ECG                  13673
Echo                  5666
Respiratory           3970
General               1877
Discharge summary     1495
Nutrition              945
Social Work            426
Rehab Services         404
Case Management        138
Consult                 35
Pharmacy                 6
Name: count, dtype: int64

## Step 6: Combine notes per admission

Multiple notes for the same hospital admission are combined into one text field so that each admission has one document for NLP modeling.

In [26]:
combined_notes = (
    first24h_notes
    .sort_values(by=["SUBJECT_ID", "HADM_ID", "NOTE_TIME"])
    .groupby(["SUBJECT_ID", "HADM_ID", "ICUSTAY_ID", "SEPSIS_LABEL"])["TEXT"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

print("Combined notes shape:", combined_notes.shape)
combined_notes.head()

Combined notes shape: (36395, 5)


,SUBJECT_ID,HADM_ID,ICUSTAY_ID,SEPSIS_LABEL,TEXT
0,3,145834.0,211552,1,[**2101-10-20**] 10:23 PM\n CHEST (PORTABLE AP...
1,4,185777.0,294638,0,[**2191-3-16**] 0500\nGeneral: Pt in to EW fro...
2,6,107064.0,228232,0,2230-0700\n\nRecieved pt from PACU following L...
3,9,150750.0,220597,0,Respiratory Care:\n Pt. intubated in EW fo...
4,11,194540.0,229441,0,[**2178-4-16**] 2:49 PM\n CTA HEAD W&W/O C & R...


## Step 7: Check class distribution

The sepsis label distribution is checked after note extraction to confirm how many sepsis and non-sepsis cases have available notes.

In [29]:
combined_notes["SEPSIS_LABEL"].value_counts()

SEPSIS_LABEL
0    32344
1     4051
Name: count, dtype: int64

## Step 8: Save extracted notes

The final first-24-hour note dataset is saved for the next notebook, which will focus on text preprocessing.

In [44]:
combined_notes.to_csv(PROCESSED_PATH + "first24h_notes.csv", index=False)

print("Saved successfully:")
print(PROCESSED_PATH + "first24h_notes.csv")

Saved successfully:
../data/processed/first24h_notes.csv


## Summary

Clinical notes were extracted from the NOTEEVENTS table and linked to the final study cohort.

The extraction process included:

- filtering notes to admissions in the final cohort
- removing error notes
- creating note timestamps
- selecting notes recorded within the first 24 hours of ICU admission
- combining multiple notes into a single admission-level document

The final dataset contains 36,395 admission-level clinical documents and 215,508 individual notes from the first 24 hours of ICU admission. The extracted notes contain no missing values and will be used for text preprocessing and NLP model development.